# Demo 2 — Farming Q&A with RAG

**FarmWise, Session 1 · Africa Digital Skills Conference**

A plain chatbot answers from memory — and can confidently make things up
(*hallucinate*). **RAG** (Retrieval-Augmented Generation) first *retrieves*
relevant text from trusted guides, then asks the model to answer **using only
that text**, with citations.

We'll build RAG one step at a time: **load → chunk → embed → store → retrieve →
prompt → answer**, then compare a plain LLM vs RAG side by side.

> Needs Ollama running with the models pulled. Run `python scripts/check_setup.py`
> first if unsure.

### Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from farmwise import config
from farmwise.llm.client import is_ollama_running

print("Ollama running:", is_ollama_running())
print("LLM model     :", config.LLM_MODEL)
print("Embedding model:", config.EMBEDDING_MODEL)

## Part A — A plain LLM, no documents

Let's ask a specific farming question with **no context**. The answer may sound
confident but be wrong or generic — we have no source to check it against.

In [ ]:
from farmwise.rag.assistant import ask_without_rag

plain = ask_without_rag("What is the critical weed-free period for maize, and why?")
print(plain.answer)

## Part B — Build RAG step by step

### Step 1 — Load the guides
Each document in `data/guides/` becomes one text with its file name as the source.

In [ ]:
from farmwise.rag.loader import load_guides

docs = load_guides()
for d in docs:
    print(f"{d.source:20s} {len(d.text):5d} chars")
print("\n--- first 250 chars of", docs[0].source, "---")
print(docs[0].text[:250])

### Step 2 — Chunk the text

Models work with small passages, not whole documents. We cut each guide into
overlapping windows of characters. The **overlap** means a sentence split across
a boundary still appears whole in one chunk.

In [ ]:
from farmwise.rag.chunker import chunk_documents

chunks = chunk_documents(docs)  # uses config.CHUNK_SIZE / CHUNK_OVERLAP
print(f"{len(docs)} documents -> {len(chunks)} chunks "
      f"(size={config.CHUNK_SIZE}, overlap={config.CHUNK_OVERLAP})")

c = chunks[0]
print(f"\nchunk 0  (source={c.source}, chars {c.start}-{c.end}):")
print(c.text)

### Step 3 — Embed: turn text into numbers

An **embedding** is a list of numbers (a vector) that captures meaning. Similar
texts get similar vectors. Let's embed one question and look at its length.

In [ ]:
from farmwise.rag.embeddings import embed_query

vec = embed_query("How much nitrogen does maize need?")
print("vector length:", len(vec))
print("first 8 numbers:", [round(x, 3) for x in vec[:8]])

**Why embeddings help:** similar meaning → closer vectors. Let's measure
similarity (cosine: 1.0 = identical meaning, 0 = unrelated).

In [ ]:
import numpy as np
from farmwise.rag.embeddings import embed_texts

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

a, b, c3 = embed_texts([
    "How do I fertilize maize?",
    "What fertilizer should I use for corn?",   # same idea, different words
    "How do I treat a sick goat?",              # unrelated
])
print("maize vs corn   (related)  :", round(cosine(a, b), 3))
print("maize vs goat   (unrelated):", round(cosine(a, c3), 3))

### Step 4 — Store the vectors

We put every chunk + its vector into a local vector database (ChromaDB). The
store is **backend-agnostic** (just `add / search / count / reset`), so Session 2
can swap in PostgreSQL + pgvector without changing the assistant.

In [ ]:
from farmwise.rag.vector_store import ChromaVectorStore

store = ChromaVectorStore()
store.reset()  # start fresh for the demo

embeddings = embed_texts([c.text for c in chunks])
store.add(
    ids=[f"{c.source}::{c.chunk_index}" for c in chunks],
    documents=[c.text for c in chunks],
    embeddings=embeddings,
    metadatas=[{"source": c.source, "chunk_index": c.chunk_index} for c in chunks],
)
print("chunks stored:", store.count())

### Step 5 — Retrieve the most relevant chunks

Embed the question, then ask the store for the `top_k` closest chunks. Higher
score = more similar.

In [ ]:
question = "How much nitrogen does maize need?"
q_vec = embed_query(question)
results = store.search(q_vec, top_k=config.TOP_K)

for r in results:
    print(f"score={r.score:.3f}  [{r.metadata['source']}]")
    print("   ", r.document[:140].replace("\n", " "), "...")
    print()

### Step 6 — Build the prompt

We paste the retrieved passages into a prompt and instruct the model to answer
**only** from them. Let's print the exact prompt the model will see.

In [ ]:
from farmwise.rag.assistant import build_prompt

prompt = build_prompt(question, results)
print(prompt)

### Step 7 — Answer, with sources

`ask()` wraps all of the above: embed → retrieve → prompt → generate, and returns
the answer plus the chunks it used as **citations**.

In [ ]:
from farmwise.rag.assistant import ask

answer = ask(question)
print("ANSWER:\n", answer.answer)
print("\nSOURCES:")
for s in answer.sources:
    print(f"  - {s.source} (chunk {s.chunk_index}, score {s.score:.3f})")

## Part C — Plain LLM vs RAG, side by side

We ask the same questions both ways. The last question is **not covered by the
guides** — RAG should say it doesn't know rather than guess.

In [ ]:
questions = [
    "What is the critical weed-free period for maize?",
    "How can I tell if my tomato plant has early blight?",
    "How do I raise tilapia fish in a pond?",   # NOT in the guides
]

rows = []
for q in questions:
    plain = ask_without_rag(q).answer.replace("\n", " ")
    rag = ask(q)
    srcs = ", ".join(sorted({s.source for s in rag.sources}))
    rows.append({
        "question": q,
        "plain LLM": plain[:160] + ("..." if len(plain) > 160 else ""),
        "RAG answer": rag.answer.replace("\n", " ")[:160],
        "sources": srcs,
    })

import pandas as pd
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows)

**Notice the third question.** The guides say nothing about fish farming, so the
RAG assistant answers *"I don't know based on the farming guides provided."* — a
grounded assistant that refuses to guess is far safer for farmers than a confident
hallucination.

## Recap
- Built RAG from scratch: load → chunk → embed → store → retrieve → prompt → answer.
- Saw embeddings capture meaning, and retrieval pull the right passages.
- Compared plain LLM vs RAG, and saw RAG cite sources and decline unknown questions.

**Next:** Demo 3 — can we actually trust these answers and predictions?